# C4 — Adaptive per-image sigma: does a training-free noise floor sharpen the detection signal?

The heteroscedastic NLL head's learned `sigma` currently does almost nothing for
detection: in `C3`/`041` the `|z|` and `raw delta` signals have near-identical AUROC
and coherence (0.60 vs 0.59; 0.26 vs 0.27) — the learned `b` has collapsed to a
near-global scale, so dividing by it barely changes the signal.

This notebook tests whether an **image-adaptive local noise floor**, computed
directly from the residual `|y - mu|` of a fixed predictor with **no training at
all**, does better. The idea (see the handoff discussion):

- Split the residual into `R_expected` (predictable local noise — texture,
  craquelure, material IR ambiguity) and `R_hidden` (the underdrawing / pentimento —
  by construction *not* predictable from RGB).
- A good `b(x)` should track `R_expected` — the spatially-varying noise floor — so
  that `z = |y - mu| / b` *up-weights* `R_hidden` against locally-noisy backgrounds.
- It should **not** track total error: if `b` predicted `|error|` perfectly the
  underdrawing would vanish from `|z|`.

**Why training-free.** The training set (especially the mockups) has RGB/IR
ambiguity but **no hidden details**; the test set has hidden details. A learned `b`
would have to transfer `R_expected` from train to test, and mockup-heavy training is
a real transfer risk. Estimating `b` per test image from its own residual sidesteps
that entirely — at the cost of mild self-absorption of the sparse `R_hidden`, which
a robust + smoothed estimator keeps small.

**Decision gate (§9).** Three questions:
1. Does `|z| adaptive` beat `raw delta` on detection (AUROC on `GT01`-`GT03`,
   coherence on all 10 `data/test/` images)?
2. Does `|z| adaptive` (same `mu`) beat `|z| learned` — i.e. is the NLL head's
   `sigma` worse than a trivial local estimate?
3. Does `adaptive` on the **deterministic** `mu` match/beat `learned` on the NLL
   `mu` — i.e. is the heteroscedastic head needed at all?

If adaptive does not beat `raw delta` either, `R_expected` is near-uniform on this
data and the honest result is: report `raw delta` / `structural delta` as the
primary signals, heteroscedastic `sigma` adds no detection value here.

Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check. `scipy.ndimage` supplies the local filters for the adaptive scale; everything else is the `C0`/`C3` toolkit.

In [ ]:
import gc

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image
from scipy.ndimage import gaussian_filter, median_filter

from scripts.calibration import (
    evaluate_calibration,
    laplace_sigma_from_scale,
    learned_zscore,
    structural_zscore,
)
from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
    pad_to_multiple,
)
from scripts.delta_analysis import analyze_delta
from scripts.detection import evaluate_detection
from scripts.stroke_stats import stroke_coherence
from scripts.trainer import load_model
from scripts.trainer_nll import load_model_nll
from scripts.visualization_nll import DEFAULT_Z_SCALE, plot_signal_comparison

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

## 1. The `data/test/` images

Every `(rgb, ir)` pair under `data/test/` — these are the paintings **with** hidden
details. The three with a hand-drawn mask (`GT01`/`GT02`/`GT03`) carry a detection
ground truth for §5; coherence (§6) runs on all of them.

In [ ]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

rgb_paths = sorted(TEST_RGB_DIR.glob("*.jpg")) + sorted(TEST_RGB_DIR.glob("*.png"))
image_pairs = [
    (p, TEST_IR_DIR / p.name)
    for p in sorted(rgb_paths)
    if (TEST_IR_DIR / p.name).exists()
]
gt_stems = {
    p.name.removesuffix("_Map.png") for p in sorted(ANNOTATIONS_DIR.glob("*_Map.png"))
}
gt_stems &= {p.stem for p, _ in image_pairs}

print(f"data/test/ images: {len(image_pairs)} | {[p.stem for p, _ in image_pairs]}")
print(f"with a ground-truth mask: {sorted(gt_stems)}")
if not image_pairs:
    raise RuntimeError("No (rgb, ir) pairs found under data/test/.")

## 2. The architecture pairs

For each architecture, the **deterministic** checkpoint (`models/deterministic/`)
supplies a pure-reconstruction `mu`; the **NLL** checkpoint (`models/nll/`) supplies
its own `mu` **and** the learned `sigma` that is the current baseline. A missing
checkpoint is skipped.

In [ ]:
ARCH_PAIRS = [
    ("unet", "unet_nll"),
    ("resunet", "resunet_nll"),
    ("attention_unet", "attention_unet_nll"),
    ("efficientnet_unet", "efficientnet_unet_nll"),
]
DET_DIR = settings.MODELS_DIR / "deterministic"
NLL_DIR = settings.MODELS_DIR / "nll"

available = []
for det_arch, nll_arch in ARCH_PAIRS:
    has_det = (DET_DIR / det_arch / "best_model.keras").exists()
    has_nll = (NLL_DIR / nll_arch / "best_model.keras").exists()
    print(f"{det_arch:<20} det={'ok' if has_det else 'MISSING':<8} "
          f"{nll_arch:<24} nll={'ok' if has_nll else 'MISSING'}")
    if has_det and has_nll:
        available.append((det_arch, nll_arch))

print(f"\nUsable pairs: {available}")
if not available:
    raise RuntimeError("No architecture has both a deterministic and an NLL checkpoint.")

## 3. The adaptive local scale

`adaptive_scale(|residual|)` estimates a spatially-varying noise floor from one
image's residual. Two methods:

- **`trimmed`** (fast default): clip `|residual|` at its own `CLIP_PCT` percentile
  to stop sparse spikes (the hidden detail) from dominating, then a Gaussian blur —
  a robust local mean, i.e. a smoothed local Laplace-scale estimate.
- **`median`**: a true local median filter (`WINDOW`) then a Gaussian blur — more
  robust, slower.
- **`mle`**: plain Gaussian-blurred `|residual|` — the local Laplace-scale MLE, *not*
  robust (absorbs `R_hidden`); included as the naive baseline.

`FLOOR` keeps `z = |r| / b` finite in perfectly-predicted regions.

In [ ]:
ADAPTIVE_METHOD = "trimmed"   # "trimmed" | "median" | "mle"
SMOOTH_SIGMA = 6.0            # Gaussian blur applied after the robust step (pixels)
WINDOW = 21                   # median-filter window (method="median" only)
CLIP_PCT = 90                 # percentile clip for method="trimmed"
FLOOR = 1e-3                  # lower bound on the returned scale


def adaptive_scale(
    resid_abs: np.ndarray,
    method: str = ADAPTIVE_METHOD,
    smooth_sigma: float = SMOOTH_SIGMA,
    window: int = WINDOW,
    clip_pct: float = CLIP_PCT,
    floor: float = FLOOR,
) -> np.ndarray:
    """A training-free, image-adaptive estimate of the local noise floor."""
    r = resid_abs.astype(np.float32)
    if method == "trimmed":
        hi = np.percentile(r, clip_pct)
        local = gaussian_filter(np.minimum(r, hi), sigma=smooth_sigma, mode="reflect")
    elif method == "median":
        local = median_filter(r, size=window, mode="reflect")
        local = gaussian_filter(local, sigma=smooth_sigma, mode="reflect")
    elif method == "mle":
        local = gaussian_filter(r, sigma=smooth_sigma, mode="reflect")
    else:
        raise ValueError(f"unknown method {method!r}")
    return np.maximum(local, floor)

## 4. Score every signal

Per architecture pair, per image, build:

| signal | `mu` | scale |
|---|---|---|
| `raw (det)` / `struct (det)` | deterministic | — |
| `|z| adapt (det)` / `sz adapt (det)` | deterministic | `adaptive_scale` |
| `|z| learned (nll)` / `sz learned (nll)` | NLL | learned `sigma` (**baseline**) |
| `|z| adapt (nll)` / `sz adapt (nll)` | NLL | `adaptive_scale` |

`sz` = structural z (`structural_zscore(structural_delta, scale)`). Models are loaded
one pair at a time and released before the next.

In [ ]:
MASK_THRESHOLD = 127


def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


def load_mask(stem: str) -> np.ndarray:
    return np.array(Image.open(ANNOTATIONS_DIR / f"{stem}_Map.png").convert("L")) > MASK_THRESHOLD


def predict_mu(model: tf.keras.Model, rgb: np.ndarray, hw: tuple[int, int], nll: bool):
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = hw
    pred = model.predict(padded[tf.newaxis, ...], verbose=0)[0, :h, :w, :]
    if nll:
        return pred[..., 0], laplace_sigma_from_scale(np.exp(pred[..., 1]))
    return pred[..., 0], None


# signal_name -> {stem: auroc}, {stem: ap}, [coherence...]
auroc: dict[tuple[str, str], dict[str, float]] = {}
ap: dict[tuple[str, str], dict[str, float]] = {}
coh: dict[tuple[str, str], list[float]] = {}

PLOT = set()  # -> set(gt_stems) to see the signal maps

for det_arch, nll_arch in available:
    print(f"\n=== {det_arch} / {nll_arch} ===")
    det_model = load_model(det_arch, model_dir=DET_DIR)
    nll_model = load_model_nll(nll_arch, model_dir=NLL_DIR, loss_name="laplace_nll",
                               beta=settings.NLL_BETA)

    for rgb_path, ir_path in image_pairs:
        stem = rgb_path.stem
        rgb, ir = load_pair(rgb_path, ir_path)
        mask = load_mask(stem) if stem in gt_stems else None

        mu_det, _ = predict_mu(det_model, rgb, ir.shape, nll=False)
        mu_nll, sigma_learned = predict_mu(nll_model, rgb, ir.shape, nll=True)

        d_det = analyze_delta(ir, mu_det)
        d_nll = analyze_delta(ir, mu_nll)
        b_det = adaptive_scale(d_det.raw_delta)
        b_nll = adaptive_scale(d_nll.raw_delta)

        signals = {
            "raw (det)": d_det.raw_delta,
            "struct (det)": d_det.structural_delta,
            "|z| adapt (det)": np.abs(learned_zscore(ir, mu_det, b_det)),
            "sz adapt (det)": structural_zscore(d_det.structural_delta, b_det),
            "|z| learned (nll)": np.abs(learned_zscore(ir, mu_nll, sigma_learned)),
            "sz learned (nll)": structural_zscore(d_nll.structural_delta, sigma_learned),
            "|z| adapt (nll)": np.abs(learned_zscore(ir, mu_nll, b_nll)),
            "sz adapt (nll)": structural_zscore(d_nll.structural_delta, b_nll),
        }

        for name, sig in signals.items():
            key = (det_arch, name)
            coh.setdefault(key, []).append(stroke_coherence(sig).coherence)
            if mask is not None:
                det = evaluate_detection(sig, mask)
                auroc.setdefault(key, {})[stem] = det.auroc
                ap.setdefault(key, {})[stem] = det.average_precision

        if stem in PLOT:
            scaled, vr = DEFAULT_Z_SCALE.apply_many(
                {k: signals[k] for k in ("|z| learned (nll)", "|z| adapt (nll)", "|z| adapt (det)")}
            )
            fig = plot_signal_comparison(ir, scaled, title=f"{det_arch} — {stem}", vrange=vr)
            plt.show()
            plt.close(fig)

        print(f"  {stem}: scored")

    del det_model, nll_model
    gc.collect()
    tf.keras.backend.clear_session()

SIGNAL_ORDER = [
    "raw (det)", "struct (det)", "|z| adapt (det)", "sz adapt (det)",
    "|z| learned (nll)", "sz learned (nll)", "|z| adapt (nll)", "sz adapt (nll)",
]
print("\ndone")

## 5. Detection AUROC, per architecture

AUROC against the hand-drawn masks, mean over `GT01`/`GT02`/`GT03`. `0.5` is chance.
The reference row is **`|z| learned (nll)`** — the signal the project uses today.
`n = 3`, so read the per-image breakdown below before trusting a delta.

In [ ]:
def auroc_mean(key):
    return float(np.mean(list(auroc[key].values()))) if key in auroc else float("nan")


if auroc:
    for det_arch, _ in available:
        base = auroc_mean((det_arch, "|z| learned (nll)"))
        print(f"\n{det_arch}   (reference '|z| learned (nll)' = {base:.4f})")
        for name in SIGNAL_ORDER:
            v = auroc_mean((det_arch, name))
            mark = "" if name == "|z| learned (nll)" else f"  ({v - base:+.4f})"
            print(f"    {name:<22} {v:.4f}{mark}")
    print(f"\n(AUROC, mean over {sorted(gt_stems)})")
else:
    print("No ground-truth mask found — section skipped.")

### Per-image AUROC breakdown

In [ ]:
if auroc:
    stems = sorted(gt_stems)
    for det_arch, _ in available:
        print(f"\n{det_arch}")
        header = "  signal".ljust(24) + "".join(s.rjust(10) for s in stems)
        print(header)
        print("  " + "-" * (len(header) - 2))
        for name in SIGNAL_ORDER:
            key = (det_arch, name)
            if key not in auroc:
                continue
            row = f"  {name}".ljust(24) + "".join(f"{auroc[key][s]:.3f}".rjust(10) for s in stems)
            print(row)
else:
    print("No ground-truth mask found — section skipped.")

## 6. Stroke coherence, per architecture

Oriented line-like structure vs. isotropic noise (`scripts.stroke_stats`), mean over
all 10 `data/test/` images — a reference-free check on §5.

In [ ]:
for det_arch, _ in available:
    base = float(np.mean(coh[(det_arch, "|z| learned (nll)")]))
    print(f"\n{det_arch}   (reference '|z| learned (nll)' = {base:.4f})")
    for name in SIGNAL_ORDER:
        vals = np.array(coh[(det_arch, name)])
        mark = "" if name == "|z| learned (nll)" else f"  ({vals.mean() - base:+.4f})"
        print(f"    {name:<22} {vals.mean():.4f} ± {vals.std():.4f}{mark}")
print(f"\n(coherence, mean ± std over {len(image_pairs)} images)")

## 7. Calibration on clean held-out data — learned `sigma` vs `adaptive`

The only honest calibration check: the validation split (part of the training pool,
so **no hidden details** — residuals there are `R_expected` only). If `adaptive` is
well-calibrated here (`z_std` near 1, low `ence`) it is modelling the noise floor
correctly; the test set cannot show this because its residuals are contaminated by
`R_hidden`.

`nll` is comparable (both Laplace). Judge `z_std` toward 1.0, `ence` down.

In [ ]:
N_CLEAN = 16
_, val_pairs, _ = mockup_aware_train_val_test_split(
    load_image_pairs(settings.IR_DIR, settings.RGB_DIR),
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)
clean_pairs = val_pairs[:N_CLEAN]
print(f"clean held-out images: {len(clean_pairs)}")

CALIB_KEYS = ["nll", "ence", "z_std", "coverage_1s", "coverage_2s", "error_sigma_spearman"]


def calib_over(model, pairs, use_adaptive):
    rows = []
    for rgb_batch, ir_batch in build_dataset(pairs, batch_size=1, augment=False, shuffle=False):
        pred = model.predict(rgb_batch, verbose=0)[0]
        ir = ir_batch[0].numpy().squeeze()
        mu = pred[..., 0]
        sigma = adaptive_scale(np.abs(ir - mu)) if use_adaptive \
            else laplace_sigma_from_scale(np.exp(pred[..., 1]))
        rows.append(evaluate_calibration(ir, mu, sigma, distribution="laplace").summary())
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}


print(f"\n{'arch / scale':<34}" + "".join(k.rjust(14) for k in CALIB_KEYS))
print("-" * (34 + 14 * len(CALIB_KEYS)))
for _, nll_arch in available:
    model = load_model_nll(nll_arch, model_dir=NLL_DIR, loss_name="laplace_nll", beta=settings.NLL_BETA)
    for label, use_adaptive in ((f"{nll_arch} learned", False), (f"{nll_arch} adaptive", True)):
        s = calib_over(model, clean_pairs, use_adaptive)
        print(f"{label:<34}" + "".join(f"{s[k]:.4f}".rjust(14) for k in CALIB_KEYS))
    del model
    gc.collect()
    tf.keras.backend.clear_session()

print("\nnominal: z_std -> 1.0, coverage 1s = 0.6827, 2s = 0.9545")

## 8. Adaptive config sensitivity (first architecture only)

`|z| adapt (nll)` AUROC (mean over `GT01`-`GT03`) across `method` x `smooth_sigma`,
on the first usable pair only — to check the §4 default is not a lucky point and to
report the best config. Re-uses the already-loaded residuals is not possible here
(models released), so this reloads one pair.

In [ ]:
if auroc and available:
    det_arch, nll_arch = available[0]
    nll_model = load_model_nll(nll_arch, model_dir=NLL_DIR, loss_name="laplace_nll", beta=settings.NLL_BETA)

    resid_by_stem, mask_by_stem = {}, {}
    for rgb_path, ir_path in image_pairs:
        if rgb_path.stem not in gt_stems:
            continue
        rgb, ir = load_pair(rgb_path, ir_path)
        mu, _ = predict_mu(nll_model, rgb, ir.shape, nll=True)
        resid_by_stem[rgb_path.stem] = (ir, mu, np.abs(ir - mu))
        mask_by_stem[rgb_path.stem] = load_mask(rgb_path.stem)
    del nll_model
    gc.collect()
    tf.keras.backend.clear_session()

    METHODS = ["trimmed", "median", "mle"]
    SIGMAS = [3.0, 6.0, 12.0]
    print(f"{det_arch}: |z| adapt (nll) AUROC, mean over {sorted(gt_stems)}\n")
    print("  method   " + "".join(f"sigma={s}".rjust(14) for s in SIGMAS))
    for method in METHODS:
        row = f"  {method:<8}"
        for sig in SIGMAS:
            aur = []
            for stem, (ir, mu, ra) in resid_by_stem.items():
                b = adaptive_scale(ra, method=method, smooth_sigma=sig)
                z = np.abs(learned_zscore(ir, mu, b))
                aur.append(evaluate_detection(z, mask_by_stem[stem]).auroc)
            row += f"{np.mean(aur):.4f}".rjust(14)
        print(row)
    print(f"\n(§4 default: method={ADAPTIVE_METHOD!r}, smooth_sigma={SMOOTH_SIGMA})")
else:
    print("No ground-truth mask — section skipped.")

## 9. Verdict — the decision gate

Per architecture, the three questions from §0, answered from §5 (AUROC) and §6
(coherence). `>` means "better by at least `EPS`".

In [ ]:
EPS = 0.010  # minimum AUROC / coherence gap to call a real difference


def better(a, b):
    return "yes" if a - b > EPS else ("~" if abs(a - b) <= EPS else "no")


print(f"{'arch':<20}{'Q1 |z|adapt>raw':<20}{'Q2 adapt>learned':<20}{'Q3 det-adapt>=nll-learned':<28}")
print("-" * 88)
for det_arch, _ in available:
    def a(name):
        return auroc_mean((det_arch, name))
    def c(name):
        return float(np.mean(coh[(det_arch, name)]))

    q1_au = better(max(a("|z| adapt (nll)"), a("|z| adapt (det)")), a("raw (det)"))
    q1_co = better(max(c("|z| adapt (nll)"), c("|z| adapt (det)")), c("raw (det)"))
    q2_au = better(a("|z| adapt (nll)"), a("|z| learned (nll)"))
    q2_co = better(c("|z| adapt (nll)"), c("|z| learned (nll)"))
    q3_au = better(a("|z| adapt (det)"), a("|z| learned (nll)") - EPS)  # ">=" -> allow tie
    q3_co = better(c("|z| adapt (det)"), c("|z| learned (nll)") - EPS)

    print(f"{det_arch:<20}"
          f"{f'AUROC {q1_au} / coh {q1_co}':<20}"
          f"{f'AUROC {q2_au} / coh {q2_co}':<20}"
          f"{f'AUROC {q3_au} / coh {q3_co}':<28}")

print(f"\n(EPS = {EPS}; Q1 uses the better of the two adaptive mu sources)")

## 10. Conclusion

_Fill in after running._

- **Q1 — does adaptive normalization beat `raw delta`?** …
- **Q2 — is the learned `sigma` worse than a trivial adaptive estimate?** …
- **Q3 — is the heteroscedastic head needed at all (deterministic `mu` + adaptive)?** …
- **§7 — is `adaptive` calibrated on clean held-out data?** …
- **§8 — best adaptive config, and is §4's default close to it?** …

**Decision:**
- If adaptive clearly wins → adopt `|z| adapt` (state which `mu` source and config)
  as the primary detection signal; the NLL head becomes optional.
- If adaptive ≈ learned ≈ raw → `R_expected` is near-uniform on this data. Report
  `raw delta` / `structural delta` as primary; note heteroscedastic `sigma` adds no
  detection value here (a valid negative result). Feeds directly into the k-fold
  model choice and the write-up of finding #8 (the heteroscedastic head's rationale
  vs. its measured payoff).